# Simple RAG Pipeline

A complete, minimal RAG (Retrieval Augmented Generation) pipeline.

This notebook demonstrates the full end-to-end RAG flow:
1. Load sample documents
2. Chunk the documents into smaller pieces
3. Store the chunks in a ChromaDB vector database (embeddings happen automatically)
4. Accept a user question
5. Retrieve the most relevant chunks via semantic search
6. Build an augmented prompt that includes the retrieved context
7. Send the prompt to Claude and get a grounded response

**Prerequisites:** Make sure you have your `ANTHROPIC_API_KEY` set as an environment variable.

**Note:** This notebook requires the `DOCUMENTS` data from `sample_documents.ipynb`. You can either:
1. Run `sample_documents.ipynb` first and copy the `DOCUMENTS` variable, or
2. The `DOCUMENTS` data is included inline below for convenience.

## Install Dependencies

In [ ]:
!pip install anthropic chromadb

## Imports

In [ ]:
import os
import textwrap

import anthropic
import chromadb

## Load Sample Documents

Import our sample knowledge base. The `DOCUMENTS` list contains policies, FAQs, product descriptions, and company info for a fictional company called Acme Corp.

**Option A:** If you have already run `sample_documents.ipynb`, you can import from there.

**Option B:** The `DOCUMENTS` data is defined inline in the cell below for standalone use.

In [ ]:
# Import our sample knowledge base
# If running standalone, this cell defines DOCUMENTS inline.
# If you prefer, you can instead run sample_documents.ipynb and import from there.

from sample_documents import DOCUMENTS

## Step 1 -- Chunking

LLMs and vector search work best with smaller, focused pieces of text. We split each document into chunks of roughly `chunk_size` characters, with an overlap so we don't accidentally cut an idea in half.

In [ ]:
def chunk_document(text: str, chunk_size: int = 500, overlap: int = 100) -> list[str]:
    """
    Split a document into overlapping chunks.

    Args:
        text:       The full document text.
        chunk_size: Target size of each chunk in characters.
        overlap:    Number of overlapping characters between consecutive chunks.

    Returns:
        A list of text chunks.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        # Move the window forward, but keep `overlap` characters of context
        start += chunk_size - overlap
    return chunks

### Prepare Chunks for ChromaDB

Take the raw documents and produce three parallel lists that ChromaDB needs:
- `ids` -- A unique ID for each chunk
- `texts` -- The chunk text (ChromaDB calls these "documents")
- `metadatas` -- Metadata attached to each chunk (source title, category, etc.)

In [ ]:
def prepare_chunks(documents: list[dict]) -> tuple[list[str], list[str], list[dict]]:
    """
    Take the raw documents and produce three parallel lists that ChromaDB needs:
      - ids:        A unique ID for each chunk
      - texts:      The chunk text (ChromaDB calls these "documents")
      - metadatas:  Metadata attached to each chunk (source title, category, etc.)
    """
    ids = []
    texts = []
    metadatas = []

    for doc_idx, doc in enumerate(documents):
        chunks = chunk_document(doc["content"])
        for chunk_idx, chunk_text in enumerate(chunks):
            # Create a unique, readable ID for every chunk
            chunk_id = f"doc{doc_idx}_chunk{chunk_idx}"
            ids.append(chunk_id)
            texts.append(chunk_text)
            metadatas.append(
                {
                    "source_title": doc["title"],
                    "category": doc["metadata"]["category"],
                    "chunk_index": chunk_idx,
                }
            )

    return ids, texts, metadatas

## Step 2 -- Build the Vector Store

ChromaDB handles embedding automatically using its built-in model. We just hand it text and it converts each chunk into a vector, stores it, and indexes it for fast similarity search.

In [ ]:
def build_vector_store(
    ids: list[str], texts: list[str], metadatas: list[dict]
) -> chromadb.Collection:
    """
    Create an in-memory ChromaDB collection and load all chunks into it.

    ChromaDB will automatically embed each chunk using its default embedding
    function (all-MiniLM-L6-v2, a lightweight sentence-transformer model).
    """
    # Create an ephemeral (in-memory) ChromaDB client — no files written to disk
    client = chromadb.Client()

    # Create (or get) a collection — think of it like a table in a database
    collection = client.create_collection(
        name="acme_knowledge_base",
        metadata={"description": "Acme Corp internal knowledge base"},
    )

    # Add all chunks to the collection.
    # ChromaDB embeds them automatically behind the scenes.
    collection.add(
        ids=ids,
        documents=texts,
        metadatas=metadatas,
    )

    print(f"  Loaded {collection.count()} chunks into the vector store.")
    return collection

## Step 3 -- Retrieve Relevant Chunks

Given a user question, we ask ChromaDB to find the most similar chunks. ChromaDB embeds the question using the same model, then does a cosine similarity search against all stored chunk vectors.

In [ ]:
def retrieve(collection: chromadb.Collection, query: str, top_k: int = 5) -> dict:
    """
    Search the vector store for chunks most relevant to the query.

    Args:
        collection: The ChromaDB collection to search.
        query:      The user's question in natural language.
        top_k:      How many chunks to retrieve.

    Returns:
        ChromaDB results dict with 'documents', 'metadatas', and 'distances'.
    """
    results = collection.query(
        query_texts=[query],
        n_results=top_k,
    )
    return results

## Step 4 -- Build the Augmented Prompt

This is the "Augmented" part of RAG. We take the retrieved chunks and inject them into the prompt so the LLM can answer based on real data.

In [ ]:
def build_prompt(query: str, retrieved_docs: list[str], sources: list[str]) -> str:
    """
    Construct the augmented prompt.

    The prompt has three parts:
      1. System instruction — tells the LLM how to behave
      2. Retrieved context — the documents we found
      3. The user's original question

    Args:
        query:          The user's question.
        retrieved_docs: List of chunk texts from the vector DB.
        sources:        List of source titles for each chunk.

    Returns:
        The complete prompt string to send to the LLM.
    """
    # Format each retrieved chunk with its source title
    context_sections = []
    for i, (doc_text, source) in enumerate(zip(retrieved_docs, sources), 1):
        context_sections.append(f"[Document {i}: {source}]\n{doc_text}")

    context_block = "\n\n---\n\n".join(context_sections)

    prompt = (
        f"Using ONLY the context provided below, answer the user's question. "
        f"If the context does not contain enough information to answer the "
        f"question, say: 'I don't have enough information to answer that.'\n\n"
        f"CONTEXT:\n\n{context_block}\n\n"
        f"QUESTION: {query}"
    )
    return prompt

## Step 5 -- Generate a Response with Claude

We send the augmented prompt to Claude. Because the prompt contains the retrieved documents, Claude's answer will be grounded in our actual data rather than its general training knowledge.

In [ ]:
def generate_response(prompt: str) -> str:
    """
    Send the augmented prompt to Claude and return the response.
    """
    # Initialize the Anthropic client (reads ANTHROPIC_API_KEY from env)
    client = anthropic.Anthropic(
        api_key=os.environ.get("ANTHROPIC_API_KEY"),
    )

    message = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )

    # Extract the text from the response
    return message.content[0].text

## Full RAG Pipeline

This function ties everything together:

**Load docs -> Chunk -> Store -> Retrieve -> Augment -> Generate**

In [ ]:
def run_rag_pipeline(question: str) -> str:
    """
    Execute the complete RAG pipeline for a given question.

    This is the function that ties everything together:
      Load docs -> Chunk -> Store -> Retrieve -> Augment -> Generate
    """
    print("=" * 60)
    print("RAG PIPELINE")
    print("=" * 60)

    # --- Phase 1: Indexing (normally done once, offline) ---
    print("\n1. Chunking documents...")
    ids, texts, metadatas = prepare_chunks(DOCUMENTS)
    print(f"  Created {len(ids)} chunks from {len(DOCUMENTS)} documents.")

    print("\n2. Building vector store...")
    collection = build_vector_store(ids, texts, metadatas)

    # --- Phase 2: Querying (happens per question) ---
    print(f"\n3. Retrieving relevant chunks for: '{question}'")
    results = retrieve(collection, question, top_k=5)

    # Pull out the retrieved documents and their source titles
    retrieved_docs = results["documents"][0]  # [0] because we sent 1 query
    retrieved_metas = results["metadatas"][0]
    sources = [meta["source_title"] for meta in retrieved_metas]

    print(f"  Found {len(retrieved_docs)} relevant chunks:")
    for i, source in enumerate(sources, 1):
        print(f"    {i}. {source}")

    print("\n4. Building augmented prompt...")
    prompt = build_prompt(question, retrieved_docs, sources)

    print("\n5. Generating response with Claude...")
    response = generate_response(prompt)

    return response

## Run the Pipeline

Check for the API key, then run the RAG pipeline on several example questions that demonstrate RAG's ability to answer from our data.

In [ ]:
# Check for API key
if not os.environ.get("ANTHROPIC_API_KEY"):
    print("ERROR: Please set the ANTHROPIC_API_KEY environment variable.")
    print("  export ANTHROPIC_API_KEY='your-key-here'")
else:
    print("API key found. Ready to run the pipeline.")

In [ ]:
# Example questions that demonstrate RAG's ability to answer from our data
example_questions = [
    "What is the return policy for electronics?",
    "How many days of PTO do employees with 4 years of experience get?",
    "What are the specs of the UltraBook 15?",
    "How does the Acme Rewards loyalty program work?",
]

In [ ]:
# Run the pipeline for each question
for question in example_questions:
    response = run_rag_pipeline(question)

    print("\n" + "-" * 60)
    print(f"QUESTION: {question}")
    print("-" * 60)
    # Wrap long lines for readability
    for line in response.split("\n"):
        print(textwrap.fill(line, width=80))
    print("\n")